In [ ]:
"""
ML label map -> OR flight path optimization

Inputs:
- label_map: HxW integer raster, e.g. from npy or single-channel png/tif
  0=background/bad, 1=good, 2=fair, 3=poor (ignored)

Outputs:
- planting_points.csv (candidate points)
- flight_path.csv (ordered route)
- flight_path.geojson (optional, for GIS)

Requires:
  pip install numpy scipy scikit-learn ortools
Optional (if you want georeferenced coordinates):
  pip install rasterio shapely geojson
"""

from __future__ import annotations

import math
import csv
import json
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
from scipy.ndimage import distance_transform_edt
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans

from ortools.constraint_solver import pywrapcp, routing_enums_pb2


# -----------------------------
# Config
# -----------------------------
GOOD_ID = 1
FAIR_ID = 2
BAD_IDS = {0, 3}  # ignored

# Pixel resolution (meters per pixel). Use your GSD.
GSD_M = 0.05  # e.g., 5 cm/pixel

# Minimum spacing between planting points (meters)
MIN_SPACING_M = 2.0

# Limit number of planting points (payload/time)
MAX_POINTS = 250

# Prefer good sites over fair sites by penalizing fair.
# (Higher penalty => fewer fair points chosen after thinning/clustering)
FAIR_PENALTY = 1.8

# Route settings
START_AT = "centroid"  # "centroid" or "first_point"
RETURN_TO_START = False  # True makes it a loop route


# -----------------------------
# Utility data structures
# -----------------------------
@dataclass
class PlantPoint:
    x_m: float
    y_m: float
    quality: str     # "good" or "fair"
    weight: float    # 1.0 for good, >1 for fair


# -----------------------------
# 1) Load label map
# -----------------------------
def load_label_map_npy(path: str) -> np.ndarray:
    arr = np.load(path)
    if arr.ndim == 3:
        arr = arr[..., 0]
    if arr.dtype != np.uint8 and arr.dtype != np.int32 and arr.dtype != np.int64:
        arr = arr.astype(np.int32)
    return arr


# -----------------------------
# 2) Convert label map -> candidate points (spacing-controlled)
# -----------------------------
def label_map_to_points(
    label_map: np.ndarray,
    gsd_m: float,
    min_spacing_m: float,
) -> List[PlantPoint]:
    """
    Creates candidate planting points by selecting pixels that are:
      - in good/fair classes, and
      - local maxima of distance to 'bad' regions (spacing proxy), and
      - spaced by at least min_spacing_m (approx).

    We use a distance transform:
      - Convert "plantable" to True, "non-plantable" to False
      - distance_transform_edt gives distance (in pixels) to nearest False
      - pick points where distance is large enough, then thin.

    This is a practical approach for early prototypes.
    """

    H, W = label_map.shape

    plantable = np.isin(label_map, [GOOD_ID, FAIR_ID])
    if plantable.sum() == 0:
        raise ValueError("No plantable pixels found (good/fair) in label_map.")

    # Distance from each plantable pixel to nearest non-plantable
    # For distance_transform_edt, distances are computed for non-zero elements to nearest zero.
    dt = distance_transform_edt(plantable)  # pixels

    min_spacing_px = max(1, int(round(min_spacing_m / gsd_m)))

    # Candidate pixels: plantable and at least min spacing away from non-plantable boundary
    candidates = np.argwhere((plantable) & (dt >= min_spacing_px))

    # If too many, subsample before thinning
    if candidates.shape[0] > 200000:
        idx = np.random.choice(candidates.shape[0], size=200000, replace=False)
        candidates = candidates[idx]

    # Greedy thinning: keep farthest-from-boundary first (more central, more robust)
    # Sort by distance transform descending
    cand_scores = dt[candidates[:, 0], candidates[:, 1]]
    order = np.argsort(-cand_scores)
    candidates = candidates[order]

    kept: List[Tuple[int, int]] = []
    grid = np.zeros((H, W), dtype=np.uint8)

    # Fast-ish spacing using marking a disk in a grid (approx).
    # For each kept point, mark a square window as occupied.
    r = min_spacing_px

    for (i, j) in candidates:
        if grid[i, j] == 1:
            continue
        kept.append((i, j))

        i0 = max(0, i - r)
        i1 = min(H, i + r + 1)
        j0 = max(0, j - r)
        j1 = min(W, j + r + 1)
        grid[i0:i1, j0:j1] = 1

        if len(kept) >= MAX_POINTS * 5:
            # keep a buffer; we’ll cluster down later
            break

    points: List[PlantPoint] = []
    for (i, j) in kept:
        cls = label_map[i, j]
        if cls == GOOD_ID:
            quality = "good"
            w = 1.0
        elif cls == FAIR_ID:
            quality = "fair"
            w = FAIR_PENALTY
        else:
            continue

        x_m = float(j * gsd_m)
        y_m = float(i * gsd_m)
        points.append(PlantPoint(x_m=x_m, y_m=y_m, quality=quality, weight=w))

    return points


# -----------------------------
# 3) Reduce candidates to a manageable set (clustering)
# -----------------------------
def reduce_points_kmeans(points: List[PlantPoint], max_points: int) -> List[PlantPoint]:
    """
    Reduce points to <= max_points using KMeans on coordinates.
    Keeps approximate spatial coverage.
    """
    if len(points) <= max_points:
        return points

    coords = np.array([(p.x_m, p.y_m) for p in points], dtype=np.float64)

    # Weighted sampling: prefer good points by oversampling them
    weights = np.array([1.0 / p.weight for p in points], dtype=np.float64)
    weights = weights / weights.sum()

    # Sample a subset for clustering to keep runtime reasonable
    sample_n = min(len(points), 5000)
    sample_idx = np.random.choice(len(points), size=sample_n, replace=False, p=weights)
    sample_coords = coords[sample_idx]

    kmeans = KMeans(n_clusters=max_points, n_init="auto", random_state=42)
    kmeans.fit(sample_coords)

    centers = kmeans.cluster_centers_  # (K,2)

    # Assign each center a "quality" based on nearest original point
    reduced: List[PlantPoint] = []
    d = cdist(centers, coords)
    nearest = np.argmin(d, axis=1)
    for k in range(centers.shape[0]):
        src = points[int(nearest[k])]
        reduced.append(
            PlantPoint(
                x_m=float(centers[k, 0]),
                y_m=float(centers[k, 1]),
                quality=src.quality,
                weight=src.weight,
            )
        )
    return reduced


# -----------------------------
# 4) Build routing cost matrix (distance + optional turn-risk proxy)
# -----------------------------
def build_cost_matrix(points: List[PlantPoint]) -> np.ndarray:
    coords = np.array([(p.x_m, p.y_m) for p in points], dtype=np.float64)
    D = cdist(coords, coords, metric="euclidean")

    # You can incorporate "quality" as a small additive cost to discourage fair points.
    # Note: Once points are chosen, route cost is mostly geometry.
    node_pen = np.array([0.0 if p.quality == "good" else 2.0 for p in points])
    # Convert node penalty into edge penalty (entering node j)
    C = D + node_pen[None, :]

    # Make integer cost matrix for OR-Tools
    scale = 1000.0  # meters -> millimeters cost units
    C_int = np.round(C * scale).astype(np.int64)
    return C_int


# -----------------------------
# 5) Solve route using OR-Tools (TSP / VRP 1 vehicle)
# -----------------------------
def solve_route_ortools(
    cost_matrix: np.ndarray,
    start_index: int = 0,
    return_to_start: bool = False,
) -> List[int]:
    n = cost_matrix.shape[0]

    # OR-Tools Routing Index Manager
    if return_to_start:
        manager = pywrapcp.RoutingIndexManager(n, 1, start_index)
    else:
        # Open route: use a "dummy end" trick by adding an extra node with zero incoming cost.
        # Simpler approach: still solve a cycle and then cut it later.
        manager = pywrapcp.RoutingIndexManager(n, 1, start_index)

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        i = manager.IndexToNode(from_index)
        j = manager.IndexToNode(to_index)
        return int(cost_matrix[i, j])

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Search parameters
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.time_limit.seconds = 10  # increase if needed

    solution = routing.SolveWithParameters(params)
    if solution is None:
        raise RuntimeError("No routing solution found.")

    # Extract route
    index = routing.Start(0)
    route = []
    while not routing.IsEnd(index):
        route.append(manager.IndexToNode(index))
        index = solution.Value(routing.NextVar(index))
    route.append(manager.IndexToNode(index))  # end (usually equals start for TSP)

    if not return_to_start:
        # Convert cycle to open path by removing the final repeated start
        if len(route) >= 2 and route[-1] == route[0]:
            route = route[:-1]

    return route


# -----------------------------
# 6) Choose a start point
# -----------------------------
def choose_start(points: List[PlantPoint], mode: str = "centroid") -> int:
    if mode == "first_point":
        return 0
    coords = np.array([(p.x_m, p.y_m) for p in points], dtype=np.float64)
    centroid = coords.mean(axis=0)
    d = np.linalg.norm(coords - centroid[None, :], axis=1)
    return int(np.argmin(d))


# -----------------------------
# 7) Export
# -----------------------------
def save_points_csv(points: List[PlantPoint], path: str) -> None:
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "x_m", "y_m", "quality", "weight"])
        for i, p in enumerate(points):
            w.writerow([i, p.x_m, p.y_m, p.quality, p.weight])


def save_route_csv(points: List[PlantPoint], route: List[int], path: str) -> None:
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["seq", "id", "x_m", "y_m", "quality"])
        for k, idx in enumerate(route):
            p = points[idx]
            w.writerow([k, idx, p.x_m, p.y_m, p.quality])


def save_route_geojson(points: List[PlantPoint], route: List[int], path: str) -> None:
    # Minimal GeoJSON without extra deps
    coords = [[points[i].x_m, points[i].y_m] for i in route]
    geo = {
        "type": "FeatureCollection",
        "features": [
            {
                "type": "Feature",
                "properties": {"name": "flight_path"},
                "geometry": {"type": "LineString", "coordinates": coords},
            }
        ],
    }
    with open(path, "w") as f:
        json.dump(geo, f, indent=2)


# -----------------------------
# Main
# -----------------------------
def main():
    label_map = load_label_map_npy("label_map.npy")  # <-- your integer mask

    # 1) Raster -> many candidate points
    candidates = label_map_to_points(
        label_map=label_map,
        gsd_m=GSD_M,
        min_spacing_m=MIN_SPACING_M,
    )
    print(f"Candidates after thinning: {len(candidates)}")

    # 2) Reduce to payload/mission limit
    points = reduce_points_kmeans(candidates, MAX_POINTS)
    print(f"Points after clustering: {len(points)}")

    # 3) Build OR cost matrix
    cost = build_cost_matrix(points)

    # 4) Solve route
    start_idx = choose_start(points, mode=START_AT)
    route = solve_route_ortools(cost, start_index=start_idx, return_to_start=RETURN_TO_START)
    print(f"Route length (nodes): {len(route)}")

    # 5) Export results
    save_points_csv(points, "planting_points.csv")
    save_route_csv(points, route, "flight_path.csv")
    save_route_geojson(points, route, "flight_path.geojson")

    print("Saved: planting_points.csv, flight_path.csv, flight_path.geojson")


if __name__ == "__main__":
    main()
